In [ ]:
NOTEBOOK_VERSION = 'v6-input-first-bootstrap'
print(f'CherryFlash {NOTEBOOK_VERSION}', flush=True)

import os
import shutil
import subprocess
import sys
from pathlib import Path

def run(cmd: list[str]) -> None:
    subprocess.run(cmd, check=True)

run(['apt-get', 'update', '-qq'])
run(['apt-get', 'install', '-y', '-qq', 'ffmpeg', 'git', 'libxrender1', 'libxi6', 'libxkbcommon-x11-0'])

blender_dir = Path('/opt/blender-4.5.4-linux-x64')
blender_bin = blender_dir / 'blender'
if not blender_bin.exists():
    run(['wget', '-q', 'https://download.blender.org/release/Blender4.5/blender-4.5.4-linux-x64.tar.xz', '-O', '/tmp/blender.tar.xz'])
    run(['tar', '-xf', '/tmp/blender.tar.xz', '-C', '/opt'])
    symlink_path = Path('/usr/local/bin/blender')
    if symlink_path.exists() or symlink_path.is_symlink():
        symlink_path.unlink()
    os.symlink(str(blender_bin), str(symlink_path))

run([sys.executable, '-m', 'pip', 'install', '-q', 'moviepy', 'pillow', 'numpy', 'imageio_ffmpeg'])
run(['blender', '--version'])


In [ ]:
import os
import json
import shutil
import subprocess
import sys
from pathlib import Path

REPO_URL = 'https://github.com/artkoder/events-bot-new'
REPO_REF = 'integration/codespace-guide-fact-first-main-20260315'

def find_scripts_root(base: Path) -> Path | None:
    direct = base / 'scripts' / 'render_mobilefeed_intro_scene1_approval.py'
    if direct.exists():
        return base
    for scripts_path in sorted(base.rglob('scripts/render_mobilefeed_intro_scene1_approval.py')):
        return scripts_path.parents[1]
    return None

NOTEBOOK_ROOT = Path.cwd().resolve()
SOURCE_FOLDER = None

for candidate in [NOTEBOOK_ROOT, Path('/kaggle/working')]:
    resolved = find_scripts_root(candidate)
    if resolved is not None:
        SOURCE_FOLDER = resolved
        print(f'Using local CherryFlash bundle at {SOURCE_FOLDER}', flush=True)
        break

if SOURCE_FOLDER is None:
    INPUT_ROOT = Path('/kaggle/input')
    top_level_dirs = sorted([p for p in INPUT_ROOT.iterdir() if p.is_dir()]) if INPUT_ROOT.exists() else []
    print('Kaggle input dirs:', [p.name for p in top_level_dirs], flush=True)
    resolved = find_scripts_root(INPUT_ROOT)
    if resolved is not None:
        SOURCE_FOLDER = resolved
        print(f'Using mounted CherryFlash bundle at {SOURCE_FOLDER}', flush=True)

if SOURCE_FOLDER is None:
    clone_root = Path('/kaggle/working/cherryflash_repo')
    if clone_root.exists():
        shutil.rmtree(clone_root)
    subprocess.run([
        'git',
        'clone',
        '--depth',
        '1',
        '--branch',
        REPO_REF,
        REPO_URL,
        str(clone_root),
    ], check=True)
    resolved = find_scripts_root(clone_root)
    if resolved is not None:
        SOURCE_FOLDER = resolved
        print(f'Using cloned CherryFlash repo at {SOURCE_FOLDER}', flush=True)

if SOURCE_FOLDER is None:
    raise RuntimeError('CherryFlash source was not found locally, under /kaggle/input, or via branch-head clone')

print(f'Using SOURCE_FOLDER={SOURCE_FOLDER}', flush=True)
os.environ['CHERRYFLASH_ROOT'] = str(SOURCE_FOLDER)
os.environ['CHERRYFLASH_ARTIFACTS_ROOT'] = '/kaggle/working/artifacts'
os.environ['BLENDER_BIN'] = shutil.which('blender') or '/usr/local/bin/blender'

script_path = SOURCE_FOLDER / 'scripts' / 'render_mobilefeed_intro_scene1_approval.py'
subprocess.run([sys.executable, str(script_path), '--final'], check=True)

artifact_dir = Path('/kaggle/working/artifacts/codex/mobilefeed_intro_scene1_final')
final_mp4 = artifact_dir / 'mobilefeed_intro_scene1_final.mp4'
if not final_mp4.exists():
    raise RuntimeError(f'Final CherryFlash approval mp4 not found: {final_mp4}')

export_mp4 = Path('/kaggle/working/cherryflash_intro_scene1_final.mp4')
shutil.copy2(final_mp4, export_mp4)

for extra_name in ['storyboard.png', 'last_intro_frame.png', 'first_scene_frame.png']:
    extra_path = artifact_dir / extra_name
    if extra_path.exists():
        shutil.copy2(extra_path, Path('/kaggle/working') / extra_name)

manifest = {
    'ok': True,
    'source_folder': str(SOURCE_FOLDER),
    'artifact_dir': str(artifact_dir),
    'export_mp4': str(export_mp4),
}
Path('/kaggle/working/cherryflash_output_manifest.json').write_text(
    json.dumps(manifest, ensure_ascii=False, indent=2),
    encoding='utf-8',
)
print(json.dumps(manifest, ensure_ascii=False, indent=2), flush=True)
